In [ ]:
# ========== 第 5 周练习：本地 RAG + Gradio ==========
# 练习目标：用 Ollama 的 nomic-embed-text 做向量、llama3.2 做生成，
# 把公司知识库文档检索后回答；前端用 Gradio 聊天。
# 依赖：本机 Ollama 已 pull 上述模型；知识库路径见下一格 folders。
# 作者注明：得到了 Claude 与课程材料的帮助。

# 基于 RAG 的 Gradio 解决方案，使用 Llama3.2 和 OLLAMA 上的 nomic-embed-text 提供相关文档中的信息
# 得到了克劳德和课程材料的帮助。


In [ ]:
# ========== 导入与全局配置：模型名 / Top-K / 知识库目录 ==========

# os：环境与路径；glob：按通配符列出知识库子目录
import os, glob
# sqlite3：把向量存进本地 SQLite（轻量 Vector Store）
import sqlite3
# json：把 embedding 列表序列化进数据库 TEXT 列
import json
# numpy：余弦相似度计算
import numpy as np
# 类型标注：List / Dict / Tuple，方便读接口
from typing import List, Dict, Tuple
# requests：HTTP 调用本机 Ollama /api/embeddings 与 /api/generate
import requests
# gradio：浏览器聊天 UI
import gradio as gr
# datetime：导入保留（本格未直接用，供后续扩展）
from datetime import datetime

# Ollama 嵌入模型名：须与 ollama pull 的名字一致
embedding_model = 'nomic-embed-text'
# Ollama 对话/生成模型名
llm_model = 'llama3.2'
# 检索时返回的近邻条数 k（RAG Dist top-k）
RagDist_k = 6
# 相对本笔记本定位 week5/knowledge-base 下各文档类型子目录
folders = glob.glob("../../week5/knowledge-base/*")
# 展示匹配到的文件夹列表，确认路径正确
folders


In [ ]:
# ========== 核心类：OllamaEmbeddings / SQLiteVectorStore / OllamaLLM / RAGSystem ==========

# OllamaEmbeddings：通过本机 Ollama HTTP API 把文本变成向量
class OllamaEmbeddings:
    """Generate embeddings using Ollama's embedding models."""

    # model 默认用全局 embedding_model；base_url 指向本机 Ollama
    def __init__(self, model: str = embedding_model, base_url: str = "http://localhost:11434"):
        # 保存模型名与服务根地址，供 embed_text 拼 URL
        self.model = model
        self.base_url = base_url

    # 单条文本 → 一条 embedding 向量
    def embed_text(self, text: str) -> List[float]:
        """Generate embedding for a single text."""
        # 打印前 70 字，换行换成 |，方便观察进度（不改文本本身）
        print('Processing', text[:70].replace('\n',' | '))
        # POST /api/embeddings：body 里 model + prompt（Ollama 嵌入接口约定）
        response = requests.post(
            f"{self.base_url}/api/embeddings",
            json={"model": self.model, "prompt": text}
        )
        # 200 则取出 JSON 里的 embedding 列表
        if response.status_code == 200:
            return response.json()["embedding"]
        else:
            # 失败时抛出，带上响应正文便于排查
            raise Exception(f"Error generating embedding: {response.text}")

    # 批量：对每条文本依次调用 embed_text（简单串行）
    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        """Generate embeddings for multiple texts."""
        return [self.embed_text(text) for text in texts]


# SQLiteVectorStore：用 SQLite 存 content + embedding JSON + metadata
class SQLiteVectorStore:
    """Vector store using SQLite for storing and retrieving document embeddings."""

    # db_path 默认 vector_store.db；check_same_thread=False 方便 Gradio 多线程读
    def __init__(self, db_path: str = "vector_store.db"):
        self.db_path = db_path
        self.conn = sqlite3.connect(db_path, check_same_thread=False)
        # 建表（若不存在）
        self._create_table()

    # 建 documents 表：content / embedding(TEXT) / metadata(TEXT) / 时间戳
    def _create_table(self):
        """Create the documents table if it doesn't exist."""
        cursor = self.conn.cursor()
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS documents (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                content TEXT NOT NULL,
                embedding TEXT NOT NULL,
                metadata TEXT,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
        """)
        self.conn.commit()

    # 批量写入：向量与元数据用 json.dumps 存成字符串
    def add_documents(self, texts: List[str], embeddings: List[List[float]],
                     metadatas: List[Dict] = None):
        """Add documents with their embeddings to the store."""
        cursor = self.conn.cursor()
        # 未给 metadata 时用空字典占位，与 texts 等长
        if metadatas is None:
            metadatas = [{}] * len(texts)

        # 逐条 INSERT
        for text, embedding, metadata in zip(texts, embeddings, metadatas):
            cursor.execute("""
                INSERT INTO documents (content, embedding, metadata)
                VALUES (?, ?, ?)
            """, (text, json.dumps(embedding), json.dumps(metadata)))

        self.conn.commit()

    # 余弦相似度：点积 / (L2 范数乘积)
    def cosine_similarity(self, vec1: np.ndarray, vec2: np.ndarray) -> float:
        """Calculate cosine similarity between two vectors."""
        return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

    # 暴力检索：读出全部文档，算相似度，取 Top-k
    def similarity_search(self, query_embedding: List[float], k: int = 3) -> List[Tuple[str, float, Dict]]:
        """Search for the k most similar documents."""
        cursor = self.conn.cursor()
        cursor.execute("SELECT content, embedding, metadata FROM documents")
        results = cursor.fetchall()

        # 查询向量转 numpy
        query_vec = np.array(query_embedding)
        similarities = []

        # 每条文档：反序列化 embedding，算相似度，连同 metadata 收集
        for content, embedding_json, metadata_json in results:
            doc_vec = np.array(json.loads(embedding_json))
            similarity = self.cosine_similarity(query_vec, doc_vec)
            similarities.append((content, similarity, json.loads(metadata_json)))

        # 按相似度排序（最高的在前）并返回前 k 个
        similarities.sort(key=lambda x: x[1], reverse=True)
        return similarities[:k]

    # 清空表内全部文档（重建索引前可用）
    def clear_all(self):
        """Clear all documents from the store."""
        cursor = self.conn.cursor()
        cursor.execute("DELETE FROM documents")
        self.conn.commit()

    # 统计当前库中文档条数
    def get_document_count(self) -> int:
        """Get the total number of documents in the store."""
        cursor = self.conn.cursor()
        cursor.execute("SELECT COUNT(*) FROM documents")
        return cursor.fetchone()[0]


# OllamaLLM：封装 /api/generate 文本生成
class OllamaLLM:
    """Interact with Ollama LLM for text generation."""

    # 默认用全局 llm_model；同样指向本机 11434
    def __init__(self, model: str = llm_model, base_url: str = "http://localhost:11434"):
        self.model = model
        self.base_url = base_url

    # prompt → 完整回答字符串；stream 参数原样传给 Ollama（此处默认 False）
    def generate(self, prompt: str, stream: bool = False) -> str:
        """Generate text from the LLM."""
        response = requests.post(
            f"{self.base_url}/api/generate",
            json={"model": self.model, "prompt": prompt, "stream": stream}
        )

        if response.status_code == 200:
            return response.json()["response"]
        else:
            raise Exception(f"Error generating response: {response.text}")


# RAGSystem：嵌入 + 向量库 + LLM 串成「入库 / 问答」
class RAGSystem:
    """RAG system combining vector store, embeddings, and LLM."""

    # 组装三件套：Embeddings、SQLite 库、LLM
    def __init__(self, embedding_model: str = embedding_model,
                 llm_model: str = llm_model,
                 db_path: str = "vector_store.db"):
        self.embeddings = OllamaEmbeddings(model=embedding_model)
        self.vector_store = SQLiteVectorStore(db_path=db_path)
        self.llm = OllamaLLM(model=llm_model)

    # 入库：抽 content/metadata → 批量 embed → 写入 SQLite
    def add_documents(self, documents: List[Dict[str, str]]):
        """
        Add documents to the RAG system.
        documents: List of dicts with 'content' and optional 'metadata'
        """
        # 取出正文列表与元数据列表
        texts = [doc['content'] for doc in documents]
        metadatas = [doc.get('metadata', {}) for doc in documents]

        print(f"Generating embeddings for {len(texts)} documents...")
        embeddings = self.embeddings.embed_documents(texts)

        print("Storing documents in vector store...")
        self.vector_store.add_documents(texts, embeddings, metadatas)
        print(f"Successfully added {len(texts)} documents!")

    # 问答：问题 embedding → Top-k 检索 → 拼上下文 prompt → LLM 生成
    def query(self, question: str, k: int = 3) -> str:
        """Query the RAG system with a question."""
        # 为查询生成嵌入
        query_embedding = self.embeddings.embed_text(question)

        # 检索相关文件
        results = self.vector_store.similarity_search(query_embedding, k=k)

        # 库空时直接返回固定英文提示（可运行字符串，不翻译）
        if not results:
            return "I don't have any information to answer this question."

        # 从检索到的文档构建上下文（带相关度分数）
        context = "\n\n".join([
            f"Document {i+1} (Relevance: {score:.2f}):\n{content}"
            for i, (content, score, _) in enumerate(results)
        ])

        # 创建 LLM 提示（system 式英文指令 + Context + Question；勿改 prompt 文案）
        prompt = f"""You are a helpful assistant answering questions based on the provided context.
            Use the following context to answer the question. If you cannot answer the question based on the context, say so.
            
            Context:
            {context}
            
            Question: {question}
            
            Answer:"""

        # 生成响应
        response = self.llm.generate(prompt)
        return response

    # 简单统计：当前库文档数
    def get_stats(self) -> str:
        """Get statistics about the RAG system."""
        doc_count = self.vector_store.get_document_count()
        return f"Total documents in database: {doc_count}"



In [ ]:
# ========== load_documents：从 folders 读 .md，打成 RAG 文档字典 ==========

def load_documents() -> List[Dict[str, str]]:
    """
    Read all files from specified folders and format them for RAG system.
    Args:
        folders: List of folder paths to read files from
    Returns:
        List of dictionaries with 'content' and 'metadata' keys
    """
    # Path：更方便判断目录/扩展名
    from pathlib import Path

    # 累积所有成功读取的文档
    documents = []
    # 本练习只索引 Markdown
    supported_extensions = {'.md'}

    # 遍历上一格 glob 得到的每个知识库子目录
    for folder in folders:
        folder_path = Path(folder)

        # 路径不存在则跳过并告警
        if not folder_path.exists():
            print(f"Warning: Folder '{folder}' does not exist. Skipping...")
            continue

        # 不是目录也跳过
        if not folder_path.is_dir():
            print(f"Warning: '{folder}' is not a directory. Skipping...")
            continue

        # 文件夹名当作文档 type（如 employees / products）
        folder_name = folder_path.name

        # 获取文件夹内所有文件（仅一层，不用 rglob）
        files = [f for f in folder_path.iterdir() if f.is_file()]

        for file_path in files:
            # 检查是否支持文件扩展名
            if file_path.suffix.lower() not in supported_extensions:
                print(f"Skipping unsupported file type: {file_path.name}")
                continue

            try:
                # 读取文件内容（UTF-8）
                with open(file_path, 'r', encoding='utf-8') as f:
                    content = f.read()

                # 创建文档字典：metadata 含 type/name/datalen，content 为全文
                document = {
                    'metadata': {
                        'type': folder_name,
                        'name': file_path.name,
                        'datalen': len(content)
                    },
                    'content': content,
                }

                documents.append(document)
                print(f"✓ Loaded: {file_path.name} from folder '{folder_name}'")

            except Exception as e:
                # 单文件失败不中断整批
                print(f"Error reading file {file_path.name}: {str(e)}")
                continue

    print(f"\nTotal documents loaded: {len(documents)}")
    return documents


In [ ]:
# ========== Gradio UI + main：加载文档、聊天、启动界面 ==========

def create_gradio_interface(rag_system: RAGSystem):
    """Create Gradio chat interface for the RAG system."""

    # 聊天回调：用户消息 → RAGSystem.query；异常时提示检查 Ollama
    def chat_fn(message, history):
        """Process chat messages."""
        try:
            # k 用全局 RagDist_k
            response = rag_system.query(message, k=RagDist_k)
            return response
        except Exception as e:
            # 错误文案保持英文（界面/排错字符串不翻译）
            return f"Error: {str(e)}\n\nMake sure Ollama is running with the required models installed."

    # 「加载文档」按钮：读磁盘 → add_documents → 返回统计
    def load_data():
        """Load sample documents into the system."""
        try:
            documents = load_documents()
            rag_system.add_documents(documents)
            stats = rag_system.get_stats()
            return f"✅ Sample documents loaded successfully!\n{stats}"
        except Exception as e:
            return f"❌ Error loading documents: {str(e)}"

    # 「统计」按钮：只读库内文档数
    def get_stats():
        """Get system statistics."""
        return rag_system.get_stats()

    # Blocks：左侧 ChatInterface，右侧加载/统计控件
    with gr.Blocks(title="RAG System - Company Knowledge Base", theme=gr.themes.Soft()) as demo:
        gr.Markdown("# 🤖 RAG System - Company Knowledge Base")
        gr.Markdown("Ask questions about company information, contracts, employees, and products.")

        with gr.Row():
            with gr.Column(scale=3):
                # ChatInterface：examples 里的问句保持英文（发给模型的示例）
                chatbot = gr.ChatInterface(
                    fn=chat_fn,
                    examples=[
                        "Who is the CTO of the company?",
                        "Who is the CEO of the company?",
                        "What products does the company offer?",
                    ],
                    title="",
                    description="💬 Chat with the company knowledge base"
                )

            with gr.Column(scale=1):
                gr.Markdown("### 📊 System Controls")
                load_btn = gr.Button("📥 Load Documents", variant="primary")
                stats_btn = gr.Button("📈 Get Statistics")
                output_box = gr.Textbox(label="System Output", lines=5)

                # 绑定按钮 → 回调 → 输出到 Textbox
                load_btn.click(fn=load_data, outputs=output_box)
                stats_btn.click(fn=get_stats, outputs=output_box)

                # 使用说明：含 ollama pull 命令（模型名用 f-string 插入变量）
                gr.Markdown(f"""
                # ##📝说明：
                1. Make sure Ollama is running
                2. Click "Load Sample Documents"
                3. Start asking questions!

                # ## 🔧 必需 Models:
                - `ollama pull {embedding_model}`
                - `ollama pull {llm_model}`
                """)

    return demo


def main():
    """Main function to run the RAG system."""
    # 启动横幅
    print("=" * 60)
    print("RAG System with Ollama and SQLite")
    print("=" * 60)

    # 初始化 RAG system（模型名与 db 路径与全局一致）
    print("\nInitializing RAG system...")
    rag_system = RAGSystem(
        embedding_model=embedding_model,
        llm_model=llm_model,
        db_path="vector_store.db"
    )

    # 提醒：先 pull 模型再问
    print("\n⚠️  Make sure Ollama is running and you have the required models:")
    print(f"   - ollama pull {embedding_model}")
    print(f"   - ollama pull {llm_model}")
    print("\nStarting Gradio interface...")

    # 创建并启动 Gradio 界面（share=False：仅本机）
    demo = create_gradio_interface(rag_system)
    demo.launch(share=False)


# 笔记本直接跑这一格即启动
main()
